# The Legacy — LoRA Finetuning on Sagemaker

This notebook finetunes Llama 3.2 1B (or 3B) with a LoRA adapter on MTG Legacy deck-building knowledge.

**Requirements:**
- Sagemaker notebook instance with GPU (ml.g5.xlarge or larger)
- Training data in `data/training/*.jsonl` (1,449 pairs)
- ~30 minutes for 1B model, ~90 minutes for 3B model

## Table of Contents
1. Setup & Dependencies
2. Load & Prepare Training Data
3. Build Evaluation Dataset
4. Load Base Model
5. Run Baseline Evaluation
6. Configure LoRA & Train
7. Run Post-Training Evaluation
8. Compare Results
9. Save & Export Model

## 1. Setup & Dependencies

In [1]:
import subprocess, sys, shutil, os

# Install training dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "peft", "trl", "datasets", "accelerate",
    "bitsandbytes", "sentencepiece", "protobuf", "huggingface_hub"])

# Nuclear fix for pyarrow binary incompatibility (conda/pip .so conflict)
# 1. Uninstall via both pip and conda
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pyarrow"], capture_output=True)
subprocess.run(["conda", "remove", "-y", "--force", "pyarrow"], capture_output=True)

# 2. Remove any leftover .so files that pip/conda missed
site_packages = os.path.join(os.path.dirname(os.__file__), "site-packages")
for name in os.listdir(site_packages):
    if name.startswith("pyarrow"):
        path = os.path.join(site_packages, name)
        print(f"Removing leftover: {path}")
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

# 3. Clean install from PyPI
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", "pyarrow==23.0.1"])

# 4. Verify — auto-restart kernel if stale binary is cached in memory
try:
    import pyarrow._acero
    print("Setup complete — pyarrow OK")
except (ValueError, AttributeError, ImportError):
    print("Restarting kernel to pick up fresh pyarrow...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

Setup complete — pyarrow OK


In [2]:
import os
import json
import torch
import glob
from pathlib import Path
from datetime import datetime

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Fix peft/torch compatibility — peft references float8 dtypes not in torch 2.6
import peft.tuners.tuners_utils as _tu
if hasattr(_tu, "UPCAST_DTYPES"):
    _tu.UPCAST_DTYPES = [d for d in _tu.UPCAST_DTYPES if hasattr(torch, d)]

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [20]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF Token: ")

HF Token:  ········


In [4]:
from huggingface_hub import login, HfApi

# Login to HuggingFace Hub (required for pushing model and accessing gated models like Llama)
# Set HF_TOKEN env var for non-interactive login, or this will prompt for a token
token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
else:
    login()

api = HfApi()
user = api.whoami()
print(f"Logged in as: {user['name']}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: malFlexion


In [5]:
# Configuration
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"  # Change to 3B if GPU allows
OUTPUT_DIR = "./lora-legacy"

# Resolve paths relative to the notebook's location (not the working directory)
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))  # fallback: cwd
# On SageMaker, notebook is in notebooks/ and data is in data/
# Walk up to repo root by looking for the data directory
for _candidate in [os.path.join(_NOTEBOOK_DIR, ".."), _NOTEBOOK_DIR, ".", ".."]:
    _data_path = os.path.join(_candidate, "data", "training")
    if os.path.isdir(_data_path):
        TRAINING_DATA_DIR = _data_path
        break
else:
    TRAINING_DATA_DIR = "./data/training"  # fallback
print(f"Training data dir: {os.path.abspath(TRAINING_DATA_DIR)}")

# HuggingFace Hub - where to push the trained LoRA adapter
HF_REPO_ID = "malhl/the-legacy-lora"  # Change to your HF username/repo
HF_PRIVATE = True                      # Keep repo private

# LoRA hyperparameters
LORA_R = 16           # Rank - higher = more capacity, more VRAM
LORA_ALPHA = 32       # Scaling factor (usually 2x rank)
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training hyperparameters
EPOCHS = 5            # 5 epochs for small dataset LoRA (model sees each example 5x)
BATCH_SIZE = 4        # Reduce to 2 if OOM
GRAD_ACCUM = 4        # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 1024
WARMUP_RATIO = 0.05

# Early stopping - stop if eval loss increases for 2 consecutive checks
EVAL_STEPS = 50       # Evaluate every 50 steps
EARLY_STOPPING_PATIENCE = 2

# Quantization (4-bit for memory efficiency)
USE_4BIT = True

Training data dir: /home/sagemaker-user/the-legacy/data/training


## 2. Load & Prepare Training Data

In [6]:
import random

def load_training_data(data_dir):
    """Load all JSONL files from the training directory."""
    all_pairs = []
    for filepath in sorted(glob.glob(os.path.join(data_dir, "*.jsonl"))):
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                pair = json.loads(line.strip())
                all_pairs.append(pair)
        print(f"  {os.path.basename(filepath)}: {sum(1 for _ in open(filepath, encoding='utf-8'))} pairs")
    print(f"\nTotal: {len(all_pairs)} training pairs")
    return all_pairs


def format_for_llama(pair, tokenizer):
    """Format a training pair as a Llama 3 chat message."""
    messages = [
        {"role": "system", "content": "You are The Legacy, an expert AI assistant for Magic: The Gathering Legacy format. You help players build decks, understand rules, evaluate cards, analyze the metagame, and improve their play. You have deep knowledge of all Legacy archetypes, card interactions, and competitive strategies."},
        {"role": "user", "content": pair["instruction"]},
        {"role": "assistant", "content": pair["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


raw_data = load_training_data(TRAINING_DATA_DIR)

# Split into train (90%) and validation (10%) for early stopping
random.seed(42)
random.shuffle(raw_data)
split_idx = int(len(raw_data) * 0.9)
train_pairs = raw_data[:split_idx]
val_pairs = raw_data[split_idx:]
print(f"\nTrain: {len(train_pairs)} pairs")
print(f"Validation: {len(val_pairs)} pairs (for early stopping)")

  board_state_analysis.jsonl: 130 pairs
  budget_substitutions.jsonl: 121 pairs
  card_evaluation.jsonl: 294 pairs
  conversation_flow.jsonl: 119 pairs
  deck_analysis.jsonl: 146 pairs
  deckbuilding_rationale.jsonl: 217 pairs
  rules_qa.jsonl: 422 pairs

Total: 1449 training pairs

Train: 1304 pairs
Validation: 145 pairs (for early stopping)


In [7]:
# Preview a few examples
for pair in raw_data[:3]:
    print(f"Q: {pair['instruction'][:80]}...")
    print(f"A: {pair['output'][:100]}...")
    print()

Q: Is Narset Parter of Veils Legacy-playable?...
A: Yes, Narset is a strong Legacy card. Her static ability limits opponents to one draw per turn, which...

Q: What does a well-constructed Legacy deck curve look like?...
A: Tempo decks peak at 1-2 mana: 20+ spells at 0-1 mana, 8-12 at 2 mana, 2-4 at 3+ mana. Control decks ...

Q: What happens if I cast a spell and my opponent casts Daze, but I want to pay the...
A: When Daze's ability triggers (counter unless you pay 1), you choose whether to pay 1 mana. If you ha...



## 3. Build Evaluation Dataset

The eval set tests 9 capabilities. These are held out from training.

In [8]:
eval_dataset = [
    # Deck legality
    {"category": "deck_legality", "instruction": "Build me a Legacy-legal Dimir Tempo deck with 60 cards main and 15 sideboard.", "expected_check": "exactly_60_main_15_side_all_legacy_legal"},
    {"category": "deck_legality", "instruction": "Is this deck Legacy-legal: 4 Brainstorm, 4 Ponder, 4 Gitaxian Probe, 4 Force of Will, 44 Islands?", "expected": "No. Gitaxian Probe is banned in Legacy."},
    {"category": "deck_legality", "instruction": "Can I play 5 copies of Lightning Bolt in my Legacy deck?", "expected": "No. Legacy allows maximum 4 copies of any card except basic lands."},
    
    # Card relevance
    {"category": "card_relevance", "instruction": "I want to build a tempo deck that beats combo. What cards should form the core?", "expected_contains": ["Force of Will", "Daze", "Thoughtseize", "Brainstorm"]},
    {"category": "card_relevance", "instruction": "What are the best sideboard cards against graveyard decks in Legacy?", "expected_contains": ["Leyline of the Void", "Surgical Extraction"]},
    {"category": "card_relevance", "instruction": "What removal spell should I play in a white Legacy deck?", "expected_contains": ["Swords to Plowshares"]},
    
    # Meta awareness
    {"category": "meta_awareness", "instruction": "What is the most played deck in Legacy right now?", "expected_contains": ["Dimir Tempo"]},
    {"category": "meta_awareness", "instruction": "What deck should I play if I want to beat Dimir Tempo?", "expected_contains": ["Lands"]},
    {"category": "meta_awareness", "instruction": "Is combo or fair strategies more popular in Legacy?", "expected_contains": ["combo", "40%"]},
    
    # Rules knowledge
    {"category": "rules_knowledge", "instruction": "Can I Daze a spell if I have no Islands in play?", "expected": "No. Daze's alternative cost requires returning an Island you control to your hand."},
    {"category": "rules_knowledge", "instruction": "Does Chalice of the Void on 1 counter Force of Will cast for its alternative cost?", "expected": "No. Force of Will has mana value 5 regardless of how it was cast. Chalice on 1 only counters spells with mana value 1."},
    {"category": "rules_knowledge", "instruction": "Can Karakas bounce Emrakul, the Aeons Torn?", "expected": "Yes. Karakas returns target legendary creature to hand. This is a land ability, not a colored spell, so Emrakul's protection does not prevent it."},
    {"category": "rules_knowledge", "instruction": "If my opponent casts Brainstorm, how many times does my Orcish Bowmasters trigger?", "expected_contains": ["two", "2", "twice"]},
    
    # Board state
    {"category": "board_state", "instruction": "I have Underground Sea, Polluted Delta, Brainstorm, Force of Will, Daze, Murktide Regent, Thoughtseize as my opening hand on the play. Should I keep?", "expected_contains": ["keep", "yes"]},
    {"category": "board_state", "instruction": "My opponent has Blood Moon in play. I have a Polluted Delta in hand. Can I use it to find an Island?", "expected_contains": ["no", "Mountain", "cannot"]},
    
    # Uniqueness (deck differs from stock)
    {"category": "uniqueness", "instruction": "Build me a unique Legacy deck that is not a copy of any existing top-tier archetype but can compete against them.", "expected_check": "not_stock_list"},
    
    # Deck analysis
    {"category": "deck_analysis", "instruction": "Identify this deck: 4 Show and Tell, 4 Sneak Attack, 3 Emrakul, 3 Atraxa, 4 Force of Will, 4 Brainstorm, 4 Ponder, 4 Lotus Petal, 3 Ancient Tomb, lands", "expected_contains": ["Sneak and Show", "combo"]},
    {"category": "deck_analysis", "instruction": "What are the strengths and weaknesses of Death and Taxes?", "expected_contains": ["Thalia", "combo", "Wasteland"]},
    
    # Budget substitutions
    {"category": "budget_subs", "instruction": "What is a budget replacement for Underground Sea?", "expected_contains": ["Watery Grave", "2 life", "shock"]},
    {"category": "budget_subs", "instruction": "What is the cheapest competitive Legacy deck?", "expected_contains": ["Oops", "Burn", "1000", "1100", "1500"]},
    
    # Card evaluation
    {"category": "card_evaluation", "instruction": "Is Counterspell good in Legacy?", "expected_contains": ["no", "slow", "Force of Will", "Daze"]},
    {"category": "card_evaluation", "instruction": "Is Orcish Bowmasters Legacy-playable?", "expected_contains": ["yes", "cantrip", "flash"]},
]

print(f"Evaluation dataset: {len(eval_dataset)} test cases")
categories = {}
for item in eval_dataset:
    cat = item["category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

Evaluation dataset: 22 test cases
  board_state: 2
  budget_subs: 2
  card_evaluation: 2
  card_relevance: 3
  deck_analysis: 2
  deck_legality: 3
  meta_awareness: 3
  rules_knowledge: 4
  uniqueness: 1


## 4. Load Base Model

In [9]:
# Quantization config for 4-bit loading
bnb_config = None
if USE_4BIT and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

Loading meta-llama/Llama-3.2-1B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded: meta-llama/Llama-3.2-1B-Instruct
Parameters: 1.2B


## 5. Run Baseline Evaluation

In [10]:
def generate_response(model, tokenizer, instruction, max_new_tokens=512):
    """Generate a response from the model for a given instruction."""
    messages = [
        {"role": "system", "content": "You are The Legacy, an expert AI assistant for Magic: The Gathering Legacy format."},
        {"role": "user", "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


def evaluate_model(model, tokenizer, eval_set, label=""):
    """Run evaluation and return scores."""
    results = []
    
    for i, item in enumerate(eval_set):
        response = generate_response(model, tokenizer, item["instruction"])
        
        # Score the response
        score = 0
        if "expected" in item:
            # Exact-ish match
            expected_lower = item["expected"].lower()
            response_lower = response.lower()
            if any(word in response_lower for word in expected_lower.split()[:5]):
                score = 1
        elif "expected_contains" in item:
            # Check if response contains expected keywords
            hits = sum(1 for kw in item["expected_contains"] if kw.lower() in response.lower())
            score = hits / len(item["expected_contains"])
        elif "expected_check" in item:
            # Manual check needed
            score = -1  # Mark for manual review
        
        results.append({
            "category": item["category"],
            "instruction": item["instruction"][:80],
            "score": score,
            "response": response[:200],
        })
        
        print(f"  [{i+1}/{len(eval_set)}] {item['category']}: score={score:.2f}")
    
    # Aggregate scores by category
    cat_scores = {}
    for r in results:
        if r["score"] >= 0:  # Skip manual-check items
            cat = r["category"]
            if cat not in cat_scores:
                cat_scores[cat] = []
            cat_scores[cat].append(r["score"])
    
    print(f"\n{'='*50}")
    print(f"Evaluation Results ({label})")
    print(f"{'='*50}")
    overall = []
    for cat, scores in sorted(cat_scores.items()):
        avg = sum(scores) / len(scores)
        overall.extend(scores)
        print(f"  {cat:25s}: {avg:.1%} ({len(scores)} tests)")
    
    total_avg = sum(overall) / len(overall) if overall else 0
    print(f"  {'OVERALL':25s}: {total_avg:.1%} ({len(overall)} tests)")
    
    return results, cat_scores, total_avg


print("Running baseline evaluation (before finetuning)...\n")
baseline_results, baseline_scores, baseline_avg = evaluate_model(
    model, tokenizer, eval_dataset, label="Baseline (no LoRA)"
)

Running baseline evaluation (before finetuning)...

  [1/22] deck_legality: score=-1.00
  [2/22] deck_legality: score=1.00
  [3/22] deck_legality: score=1.00
  [4/22] card_relevance: score=0.00
  [5/22] card_relevance: score=0.00
  [6/22] card_relevance: score=0.00
  [7/22] meta_awareness: score=0.00
  [8/22] meta_awareness: score=1.00
  [9/22] meta_awareness: score=1.00
  [10/22] rules_knowledge: score=0.00
  [11/22] rules_knowledge: score=1.00
  [12/22] rules_knowledge: score=1.00
  [13/22] rules_knowledge: score=0.33
  [14/22] board_state: score=0.50
  [15/22] board_state: score=1.00
  [16/22] uniqueness: score=-1.00
  [17/22] deck_analysis: score=0.00
  [18/22] deck_analysis: score=0.33
  [19/22] budget_subs: score=0.00
  [20/22] budget_subs: score=0.20
  [21/22] card_evaluation: score=0.25
  [22/22] card_evaluation: score=0.00

Evaluation Results (Baseline (no LoRA))
  board_state              : 75.0% (2 tests)
  budget_subs              : 10.0% (2 tests)
  card_evaluation        

## 6. Configure LoRA & Train

In [11]:
# Prepare model for training
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [12]:
# Format training and validation data
train_formatted = [{"text": format_for_llama(pair, tokenizer)} for pair in train_pairs]
val_formatted = [{"text": format_for_llama(pair, tokenizer)} for pair in val_pairs]

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"Example (first 200 chars): {train_formatted[0]['text'][:200]}...")

Train dataset: 1304 examples
Val dataset: 145 examples
Example (first 200 chars): <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 15 Apr 2026

You are The Legacy, an expert AI assistant for Magic: The Gathering Legacy f...


In [13]:
# Training arguments with early stopping
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
)

# Set max sequence length on tokenizer (compatible with all trl versions)
tokenizer.model_max_length = MAX_SEQ_LEN

# Trainer with validation set and early stopping
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f"Training config:")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"  Epochs: {EPOCHS} (with early stopping, patience={EARLY_STOPPING_PATIENCE})")
print(f"  Batch: {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}")
print(f"  Learning rate: {LEARNING_RATE}, cosine schedule")
print(f"  Eval every {EVAL_STEPS} steps, stop if val loss increases {EARLY_STOPPING_PATIENCE}x")
print(f"  Max seq length: {MAX_SEQ_LEN}")
print(f"  Train/Val split: {len(train_dataset)}/{len(val_dataset)}")

Adding EOS to train dataset:   0%|          | 0/1304 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1304 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Training config:
  Model: meta-llama/Llama-3.2-1B-Instruct
  LoRA rank: 16, alpha: 32
  Epochs: 5 (with early stopping, patience=2)
  Batch: 4x4=16
  Learning rate: 0.0002, cosine schedule
  Eval every 50 steps, stop if val loss increases 2x
  Max seq length: 1024
  Train/Val split: 1304/145


In [14]:
# Train!
print(f"Starting training at {datetime.now().strftime('%H:%M:%S')}...")
print(f"Max {EPOCHS} epochs with early stopping (patience={EARLY_STOPPING_PATIENCE})")
print(f"Will evaluate every {EVAL_STEPS} steps and stop if val loss increases.\n")

train_result = trainer.train()

print(f"\nTraining complete at {datetime.now().strftime('%H:%M:%S')}")
print(f"\nTraining metrics:")
print(f"  Final loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")

# Show training history
if hasattr(trainer.state, "log_history"):
    eval_losses = [(log["step"], log["eval_loss"]) for log in trainer.state.log_history if "eval_loss" in log]
    if eval_losses:
        print(f"\nEval loss progression:")
        for step, loss in eval_losses:
            print(f"  Step {step}: {loss:.4f}")
        best_step, best_loss = min(eval_losses, key=lambda x: x[1])
        print(f"\n  Best: step {best_step} with loss {best_loss:.4f}")
        if train_result.global_step < EPOCHS * len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):
            print(f"  Early stopping triggered (did not complete all {EPOCHS} epochs)")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting training at 01:53:42...
Max 5 epochs with early stopping (patience=2)
Will evaluate every 50 steps and stop if val loss increases.



Step,Training Loss,Validation Loss
50,1.664700,1.618737
100,1.356000,1.418776
150,1.263600,1.353283
200,1.074700,1.341153
250,1.000000,1.326369
300,0.954500,1.339102
350,0.869100,1.359007



Training complete at 03:23:43

Training metrics:
  Final loss: 1.2862
  Total steps: 350
  Runtime: 5401s

Eval loss progression:
  Step 50: 1.6187
  Step 100: 1.4188
  Step 150: 1.3533
  Step 200: 1.3412
  Step 250: 1.3264
  Step 300: 1.3391
  Step 350: 1.3590

  Best: step 250 with loss 1.3264
  Early stopping triggered (did not complete all 5 epochs)


## 7. Run Post-Training Evaluation

In [15]:
print("Running post-training evaluation...\n")
finetuned_results, finetuned_scores, finetuned_avg = evaluate_model(
    model, tokenizer, eval_dataset, label="Finetuned (with LoRA)"
)

Running post-training evaluation...

  [1/22] deck_legality: score=-1.00
  [2/22] deck_legality: score=1.00
  [3/22] deck_legality: score=1.00
  [4/22] card_relevance: score=1.00
  [5/22] card_relevance: score=0.50
  [6/22] card_relevance: score=0.00
  [7/22] meta_awareness: score=0.00
  [8/22] meta_awareness: score=0.00
  [9/22] meta_awareness: score=1.00
  [10/22] rules_knowledge: score=1.00
  [11/22] rules_knowledge: score=1.00
  [12/22] rules_knowledge: score=1.00
  [13/22] rules_knowledge: score=0.33
  [14/22] board_state: score=1.00
  [15/22] board_state: score=0.00
  [16/22] uniqueness: score=-1.00
  [17/22] deck_analysis: score=1.00
  [18/22] deck_analysis: score=0.33
  [19/22] budget_subs: score=0.00
  [20/22] budget_subs: score=0.20
  [21/22] card_evaluation: score=0.25
  [22/22] card_evaluation: score=0.33

Evaluation Results (Finetuned (with LoRA))
  board_state              : 50.0% (2 tests)
  budget_subs              : 10.0% (2 tests)
  card_evaluation          : 29.2% (2

## 8. Compare Results

In [26]:
print(f"{'='*60}")
print(f"COMPARISON: Baseline vs Finetuned")
print(f"{'='*60}")
print(f"{'Category':25s} {'Baseline':>10s} {'Finetuned':>10s} {'Change':>10s}")
print(f"{'-'*60}")

all_cats = sorted(set(list(baseline_scores.keys()) + list(finetuned_scores.keys())))
for cat in all_cats:
    b_avg = sum(baseline_scores.get(cat, [0])) / max(len(baseline_scores.get(cat, [0])), 1)
    f_avg = sum(finetuned_scores.get(cat, [0])) / max(len(finetuned_scores.get(cat, [0])), 1)
    change = f_avg - b_avg
    arrow = "+" if change > 0 else ""
    print(f"{cat:25s} {b_avg:>9.1%} {f_avg:>9.1%} {arrow}{change:>8.1%}")

print(f"{'-'*60}")
change = finetuned_avg - baseline_avg
arrow = "+" if change > 0 else ""
print(f"{'OVERALL':25s} {baseline_avg:>9.1%} {finetuned_avg:>9.1%} {arrow}{change:>8.1%}")
print(f"\nTraining improved overall score by {arrow}{change:.1%}")

COMPARISON: Baseline vs Finetuned
Category                    Baseline  Finetuned     Change
------------------------------------------------------------
board_state                   75.0%     50.0%   -25.0%
budget_subs                   10.0%     10.0%     0.0%
card_evaluation               12.5%     29.2% +   16.7%
card_relevance                 0.0%     50.0% +   50.0%
deck_analysis                 16.7%     66.7% +   50.0%
deck_legality                100.0%    100.0%     0.0%
meta_awareness                66.7%     33.3%   -33.3%
rules_knowledge               58.3%     83.3% +   25.0%
------------------------------------------------------------
OVERALL                       43.1%     54.8% +   11.7%

Training improved overall score by +11.7%


In [27]:
# Show detailed responses for manual review
print("\nDETAILED RESPONSES FOR MANUAL REVIEW")
print("=" * 60)
for b, f in zip(baseline_results, finetuned_results):
    print(f"\nQ: {b['instruction']}")
    print(f"Category: {b['category']}")
    print(f"Baseline ({b['score']:.1%}): {b['response']}")
    print(f"Finetuned ({f['score']:.1%}): {f['response']}")
    print("-" * 40)


DETAILED RESPONSES FOR MANUAL REVIEW

Q: Build me a Legacy-legal Dimir Tempo deck with 60 cards main and 15 sideboard.
Category: deck_legality
Baseline (-100.0%): I can provide you with a 60-card main deck and 15 sideboard cards for a Dimir Tempo deck. Here's a well-rounded and competitive build that should compete in the Legacy format:

**Main Deck (60 cards)*
Finetuned (-100.0%): This is Dimir Tempo with a 60-card main and 15 sideboard slots. I recommend 4 Brainstorm + 4 Ponder + 4 Force of Will + 4 Daze + 4 Thoughtseize + 4 Wasteland + 4 Murktide Regent + 4 Orcish Bowmasters 
----------------------------------------

Q: Is this deck Legacy-legal: 4 Brainstorm, 4 Ponder, 4 Gitaxian Probe, 4 Force of 
Category: deck_legality
Baseline (100.0%): To determine if your deck is Legacy-legal, I'll need to know the specific cards you're using. However, I can give you a general overview of the deck's legality.

The deck you've listed includes:

1. 4
Finetuned (100.0%): This is almost Legacy-l

## 9. Save & Export Model

In [28]:
# Save LoRA adapter locally
lora_path = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA adapter saved locally to {lora_path}")
print(f"Adapter files:")
for f in os.listdir(lora_path):
    size = os.path.getsize(os.path.join(lora_path, f))
    print(f"  {f}: {size / 1024:.1f} KB")

LoRA adapter saved locally to ./lora-legacy/lora-adapter
Adapter files:
  README.md: 5.1 KB
  adapter_model.safetensors: 44061.0 KB
  adapter_config.json: 1.1 KB
  chat_template.jinja: 3.7 KB
  tokenizer_config.json: 49.4 KB
  special_tokens_map.json: 0.3 KB
  tokenizer.json: 16806.6 KB


In [29]:
# Optional: Merge LoRA into base model and push merged model
# Useful for Ollama/GGUF export where you need a single model file

# print("Merging LoRA into base model...")
# merged_model = model.merge_and_unload()
# merged_path = os.path.join(OUTPUT_DIR, "merged-model")
# merged_model.save_pretrained(merged_path)
# tokenizer.save_pretrained(merged_path)
# print(f"Merged model saved to {merged_path}")

# Push merged model to a separate HF repo
# merged_repo = HF_REPO_ID + "-merged"
# merged_model.push_to_hub(merged_repo, private=HF_PRIVATE, commit_message="Merged LoRA model")
# tokenizer.push_to_hub(merged_repo, private=HF_PRIVATE)
# print(f"Merged model pushed to https://huggingface.co/{merged_repo}")

In [31]:
# Save evaluation results
eval_report = {
    "model": MODEL_ID,
    "training_pairs": len(raw_data),
    "epochs": EPOCHS,
    "lora_rank": LORA_R,
    "final_loss": train_result.training_loss,
    "baseline_avg": baseline_avg,
    "finetuned_avg": finetuned_avg,
    "improvement": finetuned_avg - baseline_avg,
    "baseline_by_category": {k: sum(v)/len(v) for k, v in baseline_scores.items()},
    "finetuned_by_category": {k: sum(v)/len(v) for k, v in finetuned_scores.items()},
    "baseline_responses": baseline_results,
    "finetuned_responses": finetuned_results,
}

report_path = os.path.join("", "eval_report.json")
with open(report_path, "w") as f:
    json.dump(eval_report, f, indent=2)

print(f"Evaluation report saved to {report_path}")
print(f"\nSummary:")
print(f"  Baseline: {baseline_avg:.1%}")
print(f"  Finetuned: {finetuned_avg:.1%}")
print(f"  Improvement: +{finetuned_avg - baseline_avg:.1%}")

Evaluation report saved to eval_report.json

Summary:
  Baseline: 43.1%
  Finetuned: 54.8%
  Improvement: +11.7%


## Next Steps

1. **Review the eval report** on HuggingFace Hub — check where the model improved and where it still struggles
2. **Generate targeted training data** for weak categories, retrain
3. **For Ollama deployment**:
   - Uncomment the merge cell above to create and push a merged model
   - Pull from HF: `huggingface-cli download malhl/the-legacy-lora-merged`
   - Convert to GGUF: `python llama.cpp/convert_hf_to_gguf.py merged-model --outtype q4_0`
   - Create Ollama Modelfile and `ollama create the-legacy -f Modelfile`
4. **For direct use from HuggingFace**:
   ```python
   from peft import PeftModel
   from transformers import AutoModelForCausalLM, AutoTokenizer
   base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
   model = PeftModel.from_pretrained(base, "malhl/the-legacy-lora")
   tokenizer = AutoTokenizer.from_pretrained("malhl/the-legacy-lora")
   ```
5. **Deploy** with the FastAPI wrapper from the project plan